# A2A v1.6 Tier A pilot production

This notebook runs one checkpointable 50 ns Tier A control replica per Colab session. It verifies all six staged-equilibration audits and state hashes before starting. Run every predefined replica; do not omit or replace an unfavorable result. Tier B remains locked until both controls pass at least two of three completed replicas.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q --no-cache-dir "openmm[cuda12]==8.6.0" "mdtraj==1.11.0" numpy

In [ ]:
from pathlib import Path
import json, subprocess, sys, time, openmm

REPO = Path('/content/ECMO-Research-Project')
URL = 'https://github.com/tharranb29-spec/ECMO-Research-Project.git'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(REPO)], check=True)
platforms = [openmm.Platform.getPlatform(i).getName() for i in range(openmm.Platform.getNumPlatforms())]
print('Repository:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('OpenMM', openmm.version.short_version, 'platforms:', platforms)
assert 'CUDA' in platforms, 'Reconnect to a Colab GPU runtime before continuing.'
EQUIL = Path('/content/drive/MyDrive/a2a_md_v16/tier_a_equilibration_accepted_v164')
EQUIL_SOURCES = [
    Path('/content/drive/MyDrive/a2a_md_v16/tier_a_equilibration_v164'),
    Path('/content/drive/MyDrive/a2a_md_v16/tier_a_equilibration_v164_a100_restart'),
]
RUNS = Path('/content/drive/MyDrive/a2a_md_v16/tier_a_production')
RUNS.mkdir(parents=True, exist_ok=True)

## Verify the six accepted equilibration states

This is fail-closed: all six `equilibrated_state.xml` files must be beside their audits in Drive and match the recorded SHA-256 hashes.

In [ ]:
import hashlib, shutil
sys.path.insert(0, str(REPO/'track3_a2a'))
from md_readiness_v16 import verify_equilibration_gate
gate_source = REPO/'track3_a2a/outputs/v1.6/md/tier_a_equilibration/equilibration_gate_report.json'
gate_destination = EQUIL/'equilibration_gate_report.json'
EQUIL.mkdir(parents=True, exist_ok=True)
if not gate_destination.exists():
    shutil.copy2(gate_source, gate_destination)
elif gate_destination.read_bytes() != gate_source.read_bytes():
    raise RuntimeError('Drive aggregate gate differs from the audited repository gate; preserve it for review and do not overwrite it.')
print('Aggregate gate SHA-256:', hashlib.sha256(gate_destination.read_bytes()).hexdigest())
gate_payload = json.loads(gate_destination.read_text())
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
for row in gate_payload['runs']:
    relative = Path(row['path'])
    repository_audit = REPO/'track3_a2a/outputs/v1.6/md/tier_a_equilibration'/relative
    drive_audit = EQUIL/relative
    accepted = json.loads(repository_audit.read_text())
    original_hash = accepted.get('ingestion', {}).get('source_sha256')
    source_dir = None
    for source_root in EQUIL_SOURCES:
        candidate = source_root/relative
        if candidate.is_file() and file_sha256(candidate) == original_hash:
            source_dir = candidate.parent
            break
    if source_dir is None:
        raise RuntimeError(f'No hash-matched runtime source found for {relative}')
    source_state = source_dir/'equilibrated_state.xml'
    if file_sha256(source_state) != accepted['files']['equilibrated_state.xml']:
        raise RuntimeError(f'Equilibrated state hash mismatch: {source_state}')
    drive_audit.parent.mkdir(parents=True, exist_ok=True)
    destination_state = drive_audit.parent/'equilibrated_state.xml'
    backup = drive_audit.parent/'equilibration_audit.runtime_original.json'
    staged = {backup: source_dir/'equilibration_audit.json', destination_state: source_state}
    source_state_data = source_dir/'state_data.csv'
    if source_state_data.is_file():
        staged[drive_audit.parent/'state_data.csv'] = source_state_data
    for destination, source in staged.items():
        if destination.exists() and file_sha256(destination) != file_sha256(source):
            raise RuntimeError(f'Existing canonical file differs; preserve for review: {destination}')
        if not destination.exists():
            shutil.copy2(source, destination)
    if drive_audit.exists() and file_sha256(drive_audit) != row['sha256']:
        raise RuntimeError(f'Existing canonical accepted audit differs: {drive_audit}')
    if not drive_audit.exists():
        shutil.copy2(repository_audit, drive_audit)
    print('Verified canonical production input:', relative)
gate = verify_equilibration_gate(equilibration_root=EQUIL, require_states=True)
print(gate['status'])
for row in gate['verified_runs']:
    print(row['system_id'], row['seed'], row['state_path'])

## Select exactly one production job

Use replicas 1, 2, and 3 for both systems. Re-running the same selection resumes from its 100 ps checkpoint. An A100 is preferable when available, but the frozen configuration is identical on T4.

In [ ]:
SYSTEM = '5NM4_ZMA_native'  # or '5G53_NECA_miniGs_native_nucleotide_free'
REPLICA = 1                 # 1, 2, or 3
allowed_systems = {'5NM4_ZMA_native', '5G53_NECA_miniGs_native_nucleotide_free'}
assert SYSTEM in allowed_systems and REPLICA in {1, 2, 3}
cmd = [
    'python', '-u', str(REPO/'track3_a2a/run_tier_a_production_v16.py'),
    '--system', SYSTEM, '--replica', str(REPLICA),
    '--equilibration-root', str(EQUIL), '--output-root', str(RUNS),
    '--platform', 'CUDA',
]
print('Launching:', ' '.join(cmd))
job_dir = RUNS/SYSTEM/f'replica_{REPLICA}'
job_dir.mkdir(parents=True, exist_ok=True)
status_path = job_dir/'run_status.json'
failure_path = job_dir/'failure_audit.json'
console_path = job_dir/'run_console.log'
with console_path.open('a', buffering=1) as console:
    proc = subprocess.Popen(cmd, stdout=console, stderr=subprocess.STDOUT, text=True)
    while proc.poll() is None:
        if status_path.is_file():
            status = json.loads(status_path.read_text())
            print(status.get('status'), 'step', status.get('current_step'), '/', status.get('target_steps'), 'completed_ns', status.get('completed_ns', 0), flush=True)
        else:
            print('Initializing and verifying inputs...', flush=True)
        time.sleep(60)
console_text = console_path.read_text(errors='replace')
print(console_text[-12000:])
if proc.returncode != 0:
    if failure_path.is_file():
        print('FAILURE AUDIT:', failure_path.read_text())
    raise subprocess.CalledProcessError(proc.returncode, cmd)
print(json.dumps(json.loads(status_path.read_text()), indent=2))

## Analyze only after all six production replicas finish

The analysis reports all replicas and unlocks Tier B only if both controls satisfy the frozen 2-of-3 rule.

In [ ]:
CONTROL_GATE = RUNS/'control_gate_report.json'
subprocess.run([
    'python', str(REPO/'track3_a2a/analyze_tier_a_production_v16.py'),
    '--runs-root', str(RUNS), '--output', str(CONTROL_GATE),
], check=True)
print(CONTROL_GATE.read_text())